# 04 — PEGASUS (abstractive)

**PEGASUS** (Zhang et al., 2020) è un transformer encoder-decoder pre-addestrato con un obiettivo
pensato apposta per la summarization (Gap Sentence Generation: rigenerare frasi «importanti»
mascherate). Qui usiamo il checkpoint **`google/pegasus-multi_news`**, fine-tuned **proprio su
Multi-News**: produce riassunti lunghi, in stile digest, coerenti con i riferimenti di questo
dataset. (Il default della libreria, `pegasus-xsum`, genera riassunti di una sola frase e
verrebbe penalizzato in modo fuorviante contro riferimenti da ~220 parole.)

⚠️ **Caveat di leakage**: il campione è estratto da `complete.tab`, che include la split *train*
su cui questo checkpoint è stato addestrato. I punteggi sulle righe di train sono quindi
ottimistici; il file delle metriche conserva la colonna `split` e l'aggregato riporta anche le
medie per split — per un confronto pulito guardare la media sulla sola split `test` (lo fa il
notebook [05_confronto.ipynb](05_confronto.ipynb)).

Come BART, opera **solo sul campione condiviso**; input troncato a **1024 token** (stesso limite
e stessa prassi del notebook 03, confronto equo). Su CPU ~30–90 s a esempio, con GPU pochi
secondi. Riassunti salvati incrementalmente in `results/summaries/pegasus_sample.tsv` con
**ripresa**; metriche ricalcolabili dai soli file salvati.

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

In [2]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'pegasus'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
CHECKPOINT = 'google/pegasus-multi_news'
MIN_LEN    = 32          # lunghezza minima (token) del riassunto generato
MAX_LEN    = 256         # massimo usato in fase di addestramento del checkpoint
NUM_BEAMS  = 4
LENGTH_PENALTY = 2.0     # come summ_abst_pegasus della libreria (favorisce output più lunghi)
MAX_INPUT  = 1024        # limite di input del modello

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
DEVICE = su.rileva_device()

print(f'Checkpoint : {CHECKPOINT}')
print(f'Device     : {DEVICE}')
print(f'Output     : {OUT_PATH}')

Checkpoint : google/pegasus-multi_news
Device     : cuda
Output     : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\pegasus_sample.tsv


## Generazione dei riassunti

La funzione `summ_abst_pegasus` di pyAutoSummarizer ricarica il modello **a ogni chiamata**, non
usa mai la GPU e antepone al testo il prefisso `'summarize: '` (una convenzione di T5 che PEGASUS
non ha mai visto in addestramento). Qui replichiamo i suoi parametri di generazione (beam search
a 4 beam, `length_penalty=2.0`, `early_stopping`, troncamento a 1024 token) ma: modello caricato
**una sola volta** sul `DEVICE` rilevato, **niente** prefisso `'summarize: '`, e checkpoint
`pegasus-multi_news` con `MAX_LEN=256` al posto di `pegasus-xsum`. Il separatore `` ||||| `` tra
articoli viene sostituito con un newline prima della tokenizzazione.

In [3]:
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

tokenizer = PegasusTokenizer.from_pretrained(CHECKPOINT)
modello   = PegasusForConditionalGeneration.from_pretrained(CHECKPOINT).to(DEVICE).eval()

def genera(documento):
    inputs = tokenizer(documento, max_length=MAX_INPUT, truncation=True,
                       return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = modello.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'],
                               min_length=MIN_LEN, max_length=MAX_LEN,
                               length_penalty=LENGTH_PENALTY, num_beams=NUM_BEAMS,
                               early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT, etichetta='PEGASUS ')
scrittore.chiudi()

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\antonio.girasella\.cache\huggingface\hub\models--google--pegasus-multi_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate 

PEGASUS [1] media 4.3 s/esempio (saltati 3 gia' fatti)
PEGASUS [2] media 4.5 s/esempio (saltati 3 gia' fatti)
PEGASUS [3] media 4.3 s/esempio (saltati 3 gia' fatti)
PEGASUS [10] media 4.3 s/esempio (saltati 3 gia' fatti)
PEGASUS [20] media 4.5 s/esempio (saltati 3 gia' fatti)
PEGASUS [30] media 4.3 s/esempio (saltati 3 gia' fatti)
PEGASUS [40] media 4.2 s/esempio (saltati 3 gia' fatti)
PEGASUS [50] media 4.2 s/esempio (saltati 3 gia' fatti)
PEGASUS [60] media 4.2 s/esempio (saltati 3 gia' fatti)
PEGASUS [70] media 4.2 s/esempio (saltati 3 gia' fatti)
PEGASUS [80] media 4.2 s/esempio (saltati 3 gia' fatti)
PEGASUS [90] media 4.2 s/esempio (saltati 3 gia' fatti)
PEGASUS Completato: 97 nuovi, 3 saltati, 0 errori, 415 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/pegasus_sample_per_example.csv` e `…_aggregate.json` —
l'aggregato include le medie **per split** (fondamentale qui per il caveat di leakage sulla
split train).

In [4]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'checkpoint': CHECKPOINT, 'min_len': MIN_LEN, 'max_len': MAX_LEN,
          'num_beams': NUM_BEAMS, 'length_penalty': LENGTH_PENALTY,
          'max_input_tokens': MAX_INPUT, 'device': DEVICE,
          'note': 'niente prefisso summarize:, checkpoint multi_news (leakage su split train)',
          'libreria': 'transformers (parametri di pyAutoSummarizer.summ_abst_pegasus)'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split (attenzione al leakage su train):')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\pegasus_sample_per_example.csv (100 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\pegasus_sample_aggregate.json
{
  "rouge1_f1": 0.4700259870097103,
  "rouge1_p": 0.5331074407844688,
  "rouge1_r": 0.42834574265697023,
  "rouge2_f1": 0.23028980880855876,
  "rouge2_p": 0.26259659466235113,
  "rouge2_r": 0.21028497415293082,
  "rougeL_f1": 0.28023416401242895,
  "rougeL_p": 0.3109156366607491,
  "rougeL_r": 0.2616430087985172,
  "bleu": 0.17415749623069451,
  "meteor": 0.4690451518792982,
  "parole_generate": 187.15
}

Medie per split (attenzione al leakage su train):
  test  (n=8): ROUGE-1 F1 = 0.476
  train (n=81): ROUGE-1 F1 = 0.473
  val   (n=11): ROUGE-1 F1 = 0.445


## Ispezione qualitativa

In [5]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : – "A new line has been crossed," says Rep. Matt Lesser, a Democrat running for a state Senate seat in Connecticut. "Two days after a horrific attack in Pittsburgh, the last thing we expected was to see something like this in Connecticut." Lesser is referring to a mailer sent out by Republican opponent Ed Charamut that shows Lesser, who is Jewish, holding a wad of cash in front of him with a crazed look in his eyes, the Washington Post reports. "The juxtaposition of a Jewish candidate and money i
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 